# Tolerancia a Fallos: Overhead y Umbral

**Objetivo:** calcular el overhead de qubits del surface code y entender por qué el umbral es el número mágico.

Computación tolerante a fallos (FT): ejecutar circuitos arbitrariamente largos con errores físicos < umbral p_th, pagando un overhead de qubits y operaciones pero obteniendo error lógico arbitrariamente pequeño.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Surface code: d×d qubits físicos → 1 qubit lógico (aprox.)
# Overhead: d² qubits físicos por qubit lógico
# Error lógico: P_L ≈ (p/p_th)^((d+1)/2)  — disminuye exponencialmente con d

p_th = 0.01  # umbral surface code ~1%

def logical_error_rate(p_phys, d, p_th=0.01):
    return (p_phys / p_th)**((d+1)/2)

# Para varios tamaños de código
p_range = np.linspace(0.001, 0.015, 100)
dists = [3, 5, 7, 11, 15]

plt.figure(figsize=(8,4))
for d in dists:
    P_L = logical_error_rate(p_range, d)
    plt.semilogy(p_range*100, P_L, label=f'd={d} ({d**2} qubits físicos)')
plt.axvline(p_th*100, color='k', linestyle='--', label=f'Umbral {p_th*100}%')
plt.xlabel('Error físico p (%)')
plt.ylabel('Error lógico P_L (log)')
plt.title('Surface code: error lógico vs físico')
plt.legend(fontsize=8); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('surface_code_threshold.png', dpi=80); plt.show()

In [ ]:
# Overhead de qubits para un algoritmo concreto
# Shor N=2048 bits necesita ~1000 qubits lógicos y ~10^12 puertas
n_logical = 1000
p_target_L = 1e-10  # error total del algoritmo < 10^(-10)

print('Overhead para factorizar RSA-2048 con surface code:')
for p_phys in [0.001, 0.003, 0.005]:
    # d mínimo para P_L < p_target_L / (n_logical * n_gates)
    n_gates = 1e12
    p_per_gate = p_target_L / (n_logical * n_gates)
    # P_L = (p/p_th)^((d+1)/2) < p_per_gate
    # (d+1)/2 > log(p_per_gate) / log(p/p_th)
    ratio = p_phys / p_th
    d_min = int(np.ceil(2 * np.log(p_per_gate) / np.log(ratio) - 1))
    d_min = max(d_min, 3)
    n_physical = n_logical * d_min**2
    print(f'  p_físico={p_phys*100:.1f}%: d_min={d_min}, qubits físicos~{n_physical:,}')

## Magic State Distillation

Las puertas no-Clifford (como T) no se pueden implementar directamente en FT.
Se destilan de estados "mágicos" |T⟩ = T|+⟩ usando el protocolo Reed-Muller.

In [ ]:
# Código [[15,1,3]] para distilación de magic states
# Input: 15 estados mágicos impuros con fidelidad F_in
# Output: 1 estado mágico con fidelidad F_out ≈ F_in^3 (para F_in > 0.9)

def magic_distill_fidelity(F_in, protocol='15-1'):
    if protocol == '15-1':
        return 1 - 35*(1-F_in)**3  # aproximación
    elif protocol == '7-1':
        return 1 - 7*(1-F_in)**3

F_vals = np.linspace(0.9, 1.0, 50)
plt.figure(figsize=(7,3))
for proto in ['7-1', '15-1']:
    F_out = [magic_distill_fidelity(f, proto) for f in F_vals]
    plt.plot(F_vals, F_out, label=f'Protocolo {proto}')
plt.plot(F_vals, F_vals, 'k--', label='Sin destilación')
plt.xlabel('Fidelidad entrada'); plt.ylabel('Fidelidad salida')
plt.title('Magic State Distillation'); plt.legend()
plt.tight_layout(); plt.savefig('msd.png', dpi=80); plt.show()

In [ ]:
# Resumen: estado del arte 2025-2030
print('Estado del arte FT (2026):')
print('─'*55)
print('Plataforma         p_físico  p_th   Overhead FT')
print('─'*55)
platforms = [
    ('IBM Heron r2',    0.003, 0.01, 'd~15, ~3K q/lógico'),
    ('Google Willow',   0.002, 0.01, 'd~11, ~1.5K q/lógico'),
    ('Quantinuum H2',   0.001, 0.01, 'd~7, ~500 q/lógico'),
    ('IonQ Forte',      0.001, 0.01, 'd~7, ~500 q/lógico'),
]
for name, p, pth, overhead in platforms:
    below = '✓' if p < pth else '✗'
    print(f'{name:<20} {p*100:.2f}%    {below}    {overhead}')

## Hoja de Ruta hacia FT Práctico

In [ ]:
# Timeline estimado (conservador)
timeline = [
    (2024, 'Google Willow: demostración below-threshold con 105 qubits'),
    (2025, 'IBM Condor: 1000+ qubits, d=5 surface codes físicos'),
    (2027, 'Primeros qubits lógicos operacionales (d=7-11)'),
    (2030, 'Corrección de errores útil en aplicaciones reales'),
    (2033, 'Factorización RSA-2048 (estimación optimista)'),
]
print('Hoja de ruta estimada hacia FT práctico:')
for year, event in timeline:
    print(f'  {year}: {event}')

## Próximos pasos

- **Lab:** `Cuadernos/laboratorios/09_correccion_errores_codigo_repeticion.ipynb`
- **Surface code visual:** `Cuadernos/laboratorios/14_topologia_surface_codes.ipynb`
- **Módulo:** `Tutorial/14_surface_codes_y_horizonte_fault_tolerant/README.md`
- **¡Has completado la serie de cuadernos guiados!** → Explora `Cuadernos/laboratorios/`